# Building the flights tables with MobilitySpark

This notebook builds the flights data of the book from the state vectors that the
[OpenSky Network](https://opensky-network.org/) publishes for **Sunday, March 1, 2026**, as a
sample of the table `state_vectors` of its historical database. It is the Spark counterpart of the
script `opensky_input.sql` of chapter 5 of the first volume, which builds the same tables in
MobilityDB for June 1, 2020, and the statements below follow that script one for one:

| Table | Content |
|---|---|
| `flightsinput` | The state vectors, cleaned, with their timestamps and their position |
| `flightspositions` | The position reports, once each at its time, that the aircraft can reach |
| `flightsday` | One row per aircraft, its whole day as temporal values |
| `flights` | One row per flight, i.e., per aircraft and call sign |

The temporal types and their functions come from **MobilitySpark**, whose typed SQL surface is
generated from the MEOS library: `tgeogpoint`, `tfloat`, `ttext` and `tbool` are Spark types, and
the functions keep their MobilityDB names (`tgeogpointSeq`, `atTime`, ...), except where Spark or
Flink owns the name, as `temporalUnnest` for `unnest`. The points
are built with Apache Sedona, and the tables are stored in Delta Lake.

The day is downloaded by `flights/fetch_data.sh` of this repository into
`/opt/data/flights/parquet`, 24 Parquet files of one hour each, in the folders
`hour=<Unix time of the hour>`. The appendix "The Flights Data of the Book" describes the data and
the rules applied below.

In [1]:
## Run this cell FIRST

import subprocess
import time
import sys

print("Starting Safe YARN Cluster Cleanup...")

# 1. Clean up local zombie spark references before querying YARN
if 'spark' in sys.modules or 'pyspark' in sys.modules:
    try:
        from pyspark.sql import SparkSession
        active_session = SparkSession.getActiveSession()
        if active_session:
            print("Stopping local legacy session safely...")
            active_session.stop()
    except Exception:
        pass

try:
    # 2. Query YARN for active sessions
    cmd_list = "yarn application -list -appStates RUNNING,ACCEPTED"
    result = subprocess.check_output(cmd_list, shell=True, text=True)
    
    lines = result.strip().split("\n")
    apps_killed = 0
    
    for line in lines:
        if "application_" in line:
            app_id = line.split()[0]
            print(f"Killing conflicting YARN session: {app_id}")
            subprocess.call(f"yarn application -kill {app_id}", shell=True)
            apps_killed += 1
            
    if apps_killed > 0:
        print("Waiting for resource managers to clear memory allocations...")
        time.sleep(6)
        print("Cluster environment successfully normalized!")
    else:
        print("Cluster queue is already wide open and clean.")

except Exception as e:
    print(f"YARN query bypassed: {e}")


Starting Safe YARN Cluster Cleanup...


YARN query bypassed: Command 'yarn application -list -appStates RUNNING,ACCEPTED' returned non-zero exit status 1.


In [2]:
## Run this cell SECOND

import os
from pyspark.sql import SparkSession

print("Allocating new Spark Session...")

# The master is YARN in the cluster of the chapter; SPARK_MASTER=local[*] runs the notebook on
# a single machine. The session time zone is UTC, the time zone of the OpenSky timestamps.
spark = SparkSession.builder \
    .appName("MobilitySpark-Flights") \
    .config("spark.master", os.environ.get("SPARK_MASTER", "yarn")) \
    .config("spark.submit.deployMode", "client") \
    .config("spark.executor.instances", "2") \
    .config("spark.executor.cores", "1") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "2g")) \
    .config("spark.sql.session.timeZone", "UTC") \
    .getOrCreate()

# Register the typed SQL surface of MobilitySpark: each MobilityDB type (tgeogpoint, tfloat,
# ttext, ...) is a Spark type, and each MobilityDB function is called by its MobilityDB name
spark._jvm.org.mobilitydb.spark.sql.MobilitySparkSql.registerAll(spark._jsparkSession)

print("Spark Session successfully attached to " + spark.sparkContext.master)

Allocating new Spark Session...


Spark Session successfully attached to local[*]


In [3]:
## Run this cell THIRD

from pyspark.sql import SparkSession
from IPython.core.magic import register_cell_magic


@register_cell_magic
def sql(line, cell):

    # Eliminar comentarios SQL de líneas completas
    lines = []
    for line in cell.splitlines():
        if not line.strip().startswith("#"):
            lines.append(line)

    cell = "\n".join(lines)

    # Separar sentencias
    statements = [
        s.strip()
        for s in cell.split(";")
        if s.strip()
    ]

    for statement in statements:
        print("\n==============================")
        print("RUNNING:")
        print(statement)
        print("==============================")

        result = spark.sql(statement)
        # Only attempt to call show() if the query actually returns data rows
        if result.schema:
            result.show(truncate=False)
        else:
            print("Command executed successfully.")
            

print("cell magic %%sql prepared")

cell magic %%sql prepared


In [4]:
## Run this cell FOURTH

# Remove the folders of the four tables of this notebook from the warehouse, in case a
# previous run left them without their entry in the catalog. The other tables of the
# warehouse, such as those of the Citi Bike data, are kept.

from py4j.java_gateway import java_import

java_import(spark._jvm, "org.apache.hadoop.fs.Path")

warehouse = spark.conf.get("spark.sql.warehouse.dir")
for table in ["flightsinput", "flightspositions", "flightsday", "flights"]:
    spark.sql(f"DROP TABLE IF EXISTS {table}")
    path = spark._jvm.Path(warehouse + "/" + table)
    path.getFileSystem(spark.sparkContext._jsc.hadoopConfiguration()).delete(path, True)

print("flights tables removed from the warehouse")

flights tables removed from the warehouse


## The state vectors

A state vector reports, for one aircraft identified by its transponder address `icao24`, its
position, velocity, heading, vertical rate, call sign, and altitudes at one instant, given as a
Unix timestamp. The files hold one state vector per aircraft and per second, with the columns of
the table `state_vectors`. The view `opensky` keeps the state vectors at every tenth second,
`time % 10 = 0`, the time step of the public samples of OpenSky, and reads the squawk, which the
files store as text, as an integer.

In [5]:
%%sql

CREATE OR REPLACE TEMP VIEW opensky_parquet
USING parquet OPTIONS (path 'file:///opt/data/flights/parquet');

CREATE OR REPLACE TEMP VIEW opensky AS
SELECT time, icao24, lat, lon, velocity, heading, vertrate, callsign, onground, alert, spi,
  CAST(squawk AS INT) AS squawk, baroaltitude, geoaltitude, lastposupdate, lastcontact
FROM opensky_parquet
WHERE time % 10 = 0;

SELECT * FROM opensky LIMIT 5;


RUNNING:
CREATE OR REPLACE TEMP VIEW opensky_parquet
USING parquet OPTIONS (path 'file:///opt/data/flights/parquet')


Command executed successfully.

RUNNING:
CREATE OR REPLACE TEMP VIEW opensky AS
SELECT time, icao24, lat, lon, velocity, heading, vertrate, callsign, onground, alert, spi,
  CAST(squawk AS INT) AS squawk, baroaltitude, geoaltitude, lastposupdate, lastcontact
FROM opensky_parquet
WHERE time % 10 = 0
Command executed successfully.

RUNNING:
SELECT * FROM opensky LIMIT 5


+----------+------+------------------+-----------------+------------------+-----------------+--------+--------+--------+-----+-----+------+------------+-----------+----------------+----------------+
|time      |icao24|lat               |lon              |velocity          |heading          |vertrate|callsign|onground|alert|spi  |squawk|baroaltitude|geoaltitude|lastposupdate   |lastcontact     |
+----------+------+------------------+-----------------+------------------+-----------------+--------+--------+--------+-----+-----+------+------------+-----------+----------------+----------------+
|1772342740|010264|46.282562255859375|18.60548065929878|237.72935386453042|327.1008803835521|0.0     |MSC3555 |false   |false|false|NULL  |10965.18    |11102.34   |1.772342739387E9|1.772342739834E9|
|1772341400|a45f8e|NULL              |NULL             |NULL              |NULL             |NULL    |FFT2779 |false   |false|false|NULL  |NULL        |NULL       |NULL            |1.772341395368E9|
|1772

## Cleaning

The table `flightsinput` keeps the state vectors that have a squawk, the transponder code that
identifies an aircraft on radar, and drops the aircraft that never report a position. It converts
the Unix timestamps into timestamps and builds the position as a Sedona point in WGS84.

In [6]:
%%sql

DROP TABLE IF EXISTS flightsinput;

CREATE TABLE flightsinput USING DELTA AS
WITH icao24_withnulllon AS (
  SELECT icao24 FROM opensky GROUP BY icao24 HAVING count(lon) = 0 )
SELECT time AS et, icao24, lat, lon, velocity, heading, vertrate, callsign, onground,
  alert, spi, squawk, baroaltitude, geoaltitude, lastposupdate, lastcontact,
  timestamp_seconds(time) AS etts,
  timestamp_seconds(lastposupdate) AS lastposupdatets,
  timestamp_seconds(lastcontact) AS lastcontactts,
  ST_SetSRID(ST_Point(lon, lat), 4326) AS geo
FROM opensky
WHERE squawk IS NOT NULL
  AND icao24 NOT IN (SELECT icao24 FROM icao24_withnulllon);

SELECT count(*) AS states, count(DISTINCT icao24) AS aircraft FROM flightsinput;


RUNNING:
DROP TABLE IF EXISTS flightsinput
Command executed successfully.

RUNNING:
CREATE TABLE flightsinput USING DELTA AS
WITH icao24_withnulllon AS (
  SELECT icao24 FROM opensky GROUP BY icao24 HAVING count(lon) = 0 )
SELECT time AS et, icao24, lat, lon, velocity, heading, vertrate, callsign, onground,
  alert, spi, squawk, baroaltitude, geoaltitude, lastposupdate, lastcontact,
  timestamp_seconds(time) AS etts,
  timestamp_seconds(lastposupdate) AS lastposupdatets,
  timestamp_seconds(lastcontact) AS lastcontactts,
  ST_SetSRID(ST_Point(lon, lat), 4326) AS geo
FROM opensky
WHERE squawk IS NOT NULL
  AND icao24 NOT IN (SELECT icao24 FROM icao24_withnulllon)


Command executed successfully.

RUNNING:
SELECT count(*) AS states, count(DISTINCT icao24) AS aircraft FROM flightsinput


+--------+--------+
|states  |aircraft|
+--------+--------+
|34067894|29395   |
+--------+--------+



## The position reports

A state vector repeats the last position of its aircraft until a new position report arrives, and
`lastposupdate` is the time of that report. The table `flightspositions` takes each report once,
at its time, from the first state vector that carries it, and keeps the reports that the aircraft
can reach, by the walk of `ais/ais_clean.sql` with the speed of an aircraft for that of a vessel:

* A report is kept when the last kept report could reach it at 400 m/s along the great circle,
  the distance being measured on the WGS84 spheroid by `ST_DistanceSpheroid`.
* The walk trusts its first report, so the seed is corroborated: it is the earliest of the first
  eleven reports of an aircraft that a majority of them can reach. The reports before the seed
  are walked backwards from it by the same rule.
* An aircraft whose share of reports unreachable from the report before exceeds one half is
  dropped whole.

Spark SQL has no recursive query, and the walk of an aircraft is one pass over its reports in the
order of their time: the higher-order function `aggregate` makes that pass over
`array_sort(collect_list(struct(...)))`, carrying the last kept report, the anchor, and the numbers
of the reports kept so far. `CACHE TABLE` keeps the intermediate tables in memory for the
statement that follows.

In [ ]:
%%sql

CACHE TABLE reports AS
SELECT icao24, row_number() OVER (PARTITION BY icao24 ORDER BY lastposupdate) AS rn,
  lastposupdate AS ts, lastposupdatets, lon, lat, geoaltitude, geo
FROM (
  SELECT icao24, lastposupdate, lastposupdatets, lon, lat, geoaltitude, geo,
    row_number() OVER (PARTITION BY icao24, lastposupdate ORDER BY et) AS rowno
  FROM flightsinput
  WHERE geo IS NOT NULL AND lastposupdate IS NOT NULL ) p
WHERE rowno = 1;

CACHE TABLE keepaircraft AS
SELECT icao24 FROM (
  SELECT icao24, NOT coalesce(ST_DistanceSpheroid(lag(geo) OVER w, geo) <=
    400 * (ts - lag(ts) OVER w), true) AS unreachable
  FROM reports WINDOW w AS (PARTITION BY icao24 ORDER BY rn)) s
GROUP BY icao24 HAVING avg(CAST(unreachable AS INT)) <= 0.5;

CACHE TABLE head AS
SELECT r.icao24, r.rn, r.ts, r.geo
FROM reports r JOIN keepaircraft k ON k.icao24 = r.icao24
WHERE r.rn <= 11;

CACHE TABLE seed AS
SELECT icao24, coalesce(min(rn) FILTER (WHERE ok * 2 >= others), 1) AS seedrn
FROM (
  SELECT a.icao24, a.rn,
    count(*) FILTER (WHERE b.rn <> a.rn AND b.ts <> a.ts
      AND ST_DistanceSpheroid(a.geo, b.geo) <= 400 * abs(b.ts - a.ts)) AS ok,
    count(*) - 1 AS others
  FROM head a JOIN head b ON b.icao24 = a.icao24
  GROUP BY a.icao24, a.rn) a
GROUP BY icao24;

DROP TABLE IF EXISTS flightspositions;

CREATE TABLE flightspositions USING DELTA AS
WITH tracks AS (
  SELECT r.icao24, s.seedrn,
    array_sort(collect_list(struct(r.rn, r.ts, r.lon, r.lat))) AS reports
  FROM reports r JOIN seed s ON s.icao24 = r.icao24
  GROUP BY r.icao24, s.seedrn ),
walks AS (
  SELECT icao24,
    aggregate(filter(reports, x -> x.rn > seedrn),
      named_struct('ts', reports[seedrn - 1].ts, 'lon', reports[seedrn - 1].lon,
        'lat', reports[seedrn - 1].lat, 'kept', array(seedrn)),
      (a, x) -> CASE
        WHEN ST_DistanceSpheroid(ST_Point(a.lon, a.lat), ST_Point(x.lon, x.lat)) <=
          400 * (x.ts - a.ts)
        THEN named_struct('ts', x.ts, 'lon', x.lon, 'lat', x.lat,
          'kept', array_append(a.kept, x.rn))
        ELSE a END).kept AS forward,
    aggregate(reverse(filter(reports, x -> x.rn < seedrn)),
      named_struct('ts', reports[seedrn - 1].ts, 'lon', reports[seedrn - 1].lon,
        'lat', reports[seedrn - 1].lat, 'kept', CAST(array() AS ARRAY<INT>)),
      (a, x) -> CASE
        WHEN ST_DistanceSpheroid(ST_Point(a.lon, a.lat), ST_Point(x.lon, x.lat)) <=
          400 * (a.ts - x.ts)
        THEN named_struct('ts', x.ts, 'lon', x.lon, 'lat', x.lat,
          'kept', array_append(a.kept, x.rn))
        ELSE a END).kept AS backward
  FROM tracks ),
kept AS (
  SELECT icao24, explode(concat(forward, backward)) AS rn
  FROM walks )
SELECT r.icao24, r.lastposupdatets AS posts, r.lon, r.lat, r.geoaltitude, r.geo
FROM reports r JOIN kept k ON k.icao24 = r.icao24 AND k.rn = r.rn;

SELECT (SELECT count(*) FROM reports) AS reports, count(*) AS keptreports,
  count(DISTINCT icao24) AS aircraft
FROM flightspositions;

## The day of each aircraft

The table `flightsday` holds, for each aircraft, its trajectory as a `tgeogpoint` and its
velocity, heading, vertical rate, call sign, alert flag, and geometric altitude as temporal values.
As in MobilityDB, an instant is made of a value and a timestamp, the instants of an aircraft are
gathered in the order of their timestamps, and a sequence is built from them:

* The trajectory is made of the position reports of `flightspositions`, at the times of the
  reports. The other attributes keep the time of their state vector.
* Sedona builds the point and MEOS reads it, as MobilityDB reads a PostGIS point: Sedona writes
  it as hexadecimal extended well-known binary, which keeps its SRID, and
  `geographyFromHexEWKB` reads it as a geography, `geographyFromHexEWKB(ST_AsHEXEWKB(geo))`.
* Spark has no ordered `array_agg`: `array_sort(collect_list(struct(etts, value)))` orders the
  pairs by their timestamp, and `.value` keeps their values in that order.
* For an aircraft that never reports an attribute, `collect_list` answers an empty array where
  `array_agg` in PostgreSQL answers `NULL`. Since a sequence needs at least one instant, a `CASE`
  keeps such an attribute `NULL`. An aircraft without a kept position report has no row in
  `trajectories`, and the `LEFT JOIN` keeps its trajectory `NULL`.

In [ ]:
%%sql

DROP TABLE IF EXISTS flightsday;

CREATE TABLE flightsday USING DELTA AS
WITH trajectories AS (
  SELECT icao24,
    tgeogpointSeq(array_sort(collect_list(struct(posts,
      tgeogpoint(geographyFromHexEWKB(ST_AsHEXEWKB(geo)), posts) AS geo))).geo) AS flight
  FROM flightspositions
  GROUP BY icao24 ),
instants AS (
  SELECT icao24, etts,
    tfloat(velocity, etts) AS velocity, tfloat(heading, etts) AS heading,
    tfloat(vertrate, etts) AS vertrate, ttext(trim(callsign), etts) AS callsign,
    tbool(alert, etts) AS alert, tfloat(geoaltitude, etts) AS geoaltitude
  FROM flightsinput ),
arrays AS (
  SELECT icao24,
    array_sort(collect_list(struct(etts, velocity)) FILTER (WHERE velocity IS NOT NULL)).velocity AS velocity,
    array_sort(collect_list(struct(etts, heading)) FILTER (WHERE heading IS NOT NULL)).heading AS heading,
    array_sort(collect_list(struct(etts, vertrate)) FILTER (WHERE vertrate IS NOT NULL)).vertrate AS vertrate,
    array_sort(collect_list(struct(etts, callsign)) FILTER (WHERE callsign IS NOT NULL)).callsign AS callsign,
    array_sort(collect_list(struct(etts, alert)) FILTER (WHERE alert IS NOT NULL)).alert AS alert,
    array_sort(collect_list(struct(etts, geoaltitude)) FILTER (WHERE geoaltitude IS NOT NULL)).geoaltitude AS geoaltitude
  FROM instants
  GROUP BY icao24 ),
attributes AS (
  SELECT icao24,
    CASE WHEN size(velocity) > 0 THEN tfloatSeq(velocity) END AS velocity,
    CASE WHEN size(heading) > 0 THEN tfloatSeq(heading) END AS heading,
    CASE WHEN size(vertrate) > 0 THEN tfloatSeq(vertrate) END AS vertrate,
    CASE WHEN size(callsign) > 0 THEN ttextSeq(callsign) END AS callsign,
    CASE WHEN size(alert) > 0 THEN tboolSeq(alert) END AS alert,
    CASE WHEN size(geoaltitude) > 0 THEN tfloatSeq(geoaltitude) END AS geoaltitude
  FROM arrays )
SELECT a.icao24, t.flight, a.velocity, a.heading, a.vertrate, a.callsign, a.alert,
  a.geoaltitude
FROM attributes a LEFT JOIN trajectories t ON t.icao24 = a.icao24;

SELECT icao24, numInstants(flight) AS instants, asText(callsign) AS callsign
FROM flightsday LIMIT 5;

## The flights

An aircraft flies several flights in a day, each under its own call sign. The function
`temporalUnnest`, the `unnest` of MobilityDB, splits the temporal call sign of an aircraft into one
pair (value, time) per distinct call sign, where the time is the set of periods during which the
aircraft uses it, and `inline` turns the pairs into rows. The function `atTime` then restricts each
temporal attribute of the aircraft to the periods of the flight.

In [8]:
%%sql

DROP TABLE IF EXISTS flights;

CREATE TABLE flights USING DELTA AS
SELECT f.icao24, r.value AS callsign, r.time AS flightperiod,
  atTime(f.flight, r.time) AS flight, atTime(f.velocity, r.time) AS velocity,
  atTime(f.heading, r.time) AS heading, atTime(f.vertrate, r.time) AS vertrate,
  atTime(f.alert, r.time) AS alert, atTime(f.geoaltitude, r.time) AS geoaltitude
FROM flightsday f
LATERAL VIEW inline(temporalUnnest(f.callsign)) r AS value, time;

SELECT icao24, callsign, asText(flightperiod) AS flightperiod,
  numInstants(flight) AS instants
FROM flights LIMIT 5;


RUNNING:
DROP TABLE IF EXISTS flights
Command executed successfully.

RUNNING:
CREATE TABLE flights USING DELTA AS
SELECT f.icao24, r.value AS callsign, r.time AS flightperiod,
  atTime(f.flight, r.time) AS flight, atTime(f.velocity, r.time) AS velocity,
  atTime(f.heading, r.time) AS heading, atTime(f.vertrate, r.time) AS vertrate,
  atTime(f.alert, r.time) AS alert, atTime(f.geoaltitude, r.time) AS geoaltitude
FROM flightsday f
LATERAL VIEW inline(unnest(f.callsign)) r AS value, time


Command executed successfully.

RUNNING:
SELECT icao24, callsign, asText(flightperiod) AS flightperiod,
  numInstants(flight) AS instants
FROM flights LIMIT 5


+------+--------+--------------------------------------------------+--------+
|icao24|callsign|flightperiod                                      |instants|
+------+--------+--------------------------------------------------+--------+
|010160|MSR745  |{[2026-03-01 21:56:10+00, 2026-03-01 23:39:20+00]}|224     |
|010160|MSR793  |{[2026-03-01 13:02:10+00, 2026-03-01 17:29:40+00)}|276     |
|010160|MSR794  |{[2026-03-01 17:29:40+00, 2026-03-01 21:56:10+00)}|290     |
|010160|MSR842  |{[2026-03-01 07:28:40+00, 2026-03-01 13:02:10+00)}|2       |
|0101b8|MSR024  |{[2026-03-01 13:59:10+00, 2026-03-01 18:05:00+00)}|29      |
+------+--------+--------------------------------------------------+--------+



## Summary of the tables

In [9]:
%%sql

SELECT count(*) AS aircraft, count(flight) AS with_flight,
  sum(numInstants(flight)) AS instants
FROM flightsday;

SELECT count(*) AS flights, count(DISTINCT icao24) AS aircraft,
  count(DISTINCT callsign) AS callsigns,
  count(*) FILTER (WHERE callsign = '') AS without_callsign
FROM flights;


RUNNING:
SELECT count(*) AS aircraft, count(flight) AS with_flight,
  sum(numInstants(flight)) AS instants
FROM flightsday


+--------+-----------+--------+
|aircraft|with_flight|instants|
+--------+-----------+--------+
|29395   |29212      |27778407|
+--------+-----------+--------+


RUNNING:
SELECT count(*) AS flights, count(DISTINCT icao24) AS aircraft,
  count(DISTINCT callsign) AS callsigns,
  count(*) FILTER (WHERE callsign = '') AS without_callsign
FROM flights


+-------+--------+---------+----------------+
|flights|aircraft|callsigns|without_callsign|
+-------+--------+---------+----------------+
|68371  |29252   |65007    |219             |
+-------+--------+---------+----------------+

